# Memory Concepts (메모리 개념)

**메모리(Memory)** 는 AI 에이전트가 대화 이력과 상태를 저장하고 관리하는 시스템입니다.

메모리는 두 가지 주요 유형으로 분류됩니다:
- **단기 메모리(Short-term Memory)**: 하나의 대화 세션(thread) 내에서 이전 상호작용을 기억
- **장기 메모리(Long-term Memory)**: 여러 대화 세션에 걸쳐 정보를 유지

이 노트북에서는 메모리의 기본 개념과 사용법을 다룹니다.

**참고**: [LangChain 공식 문서 - Memory](https://docs.langchain.com/oss/python/concepts/memory)

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5.4-mini", model_provider="openai")

## 1. 단기 메모리 (Short-term Memory)

단기 메모리는 하나의 대화 세션(thread) 내에서 이전 메시지를 기억합니다.
에이전트가 대화의 문맥을 유지하고 이전 대화 내용을 참조할 수 있게 합니다.

### 1.1 Checkpointer 없이 사용

Checkpointer 없이는 각 호출이 독립적이며, 이전 대화를 기억하지 못합니다.

In [3]:
from langchain.agents import create_agent

# Checkpointer 없는 에이전트
agent_no_memory = create_agent(model, tools=[])

# 첫 번째 대화
result1 = agent_no_memory.invoke({
    "messages": [{"role": "user", "content": "내 이름은 홍길동이야."}]
})
print("응답 1:", result1["messages"][-1].content)

# 두 번째 대화 - 이전 내용을 기억하지 못함
result2 = agent_no_memory.invoke({
    "messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]
})
print("\n응답 2:", result2["messages"][-1].content)

응답 1: 반가워요, 홍길동님!  
무엇을 도와드릴까요?

응답 2: 제가 아직 당신의 이름을 알지 못해요.  
이름을 알려주시면 기억해둘게요.


### 1.2 InMemorySaver로 단기 메모리 추가

`InMemorySaver`는 메모리에 대화 이력을 저장합니다.
프로세스가 종료되면 데이터가 사라집니다.

In [4]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver를 사용한 에이전트
agent_with_memory = create_agent(
    model,
    tools=[],
    checkpointer=InMemorySaver()
)

# 대화 설정 (thread_id로 대화 식별)
config = {"configurable": {"thread_id": "conversation_1"}}

# 첫 번째 대화
result1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 홍길동이야."}]},
    config
)
print("응답 1:", result1["messages"][-1].content)

# 두 번째 대화 - 같은 thread_id로 이전 내용을 기억
result2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    config
)
print("\n응답 2:", result2["messages"][-1].content)

응답 1: 반가워요, 홍길동님! 무엇을 도와드릴까요?

응답 2: 당신의 이름은 **홍길동**입니다.


## 2. 영구 저장 (Persistent Storage)

`SqliteSaver`를 사용하면 대화 이력을 데이터베이스에 영구 저장할 수 있습니다.
프로세스가 재시작되어도 데이터가 유지됩니다.

### 2.1 SqliteSaver 사용

In [6]:
import os
from langgraph.checkpoint.sqlite import SqliteSaver

# SQLite 데이터베이스 파일 경로
DB_PATH = "memory_checkpoints.db"

for f in ["memory_checkpoints.db", "memory_checkpoints.db-wal", "memory_checkpoints.db-shm"]:
    if os.path.exists(f):
        os.remove(f)

# SqliteSaver 생성 및 사용
with SqliteSaver.from_conn_string(DB_PATH) as checkpointer:
    # 테이블 자동 생성
    checkpointer.setup()
    
    # 에이전트 생성
    agent_persistent = create_agent(
        model,
        tools=[],
        checkpointer=checkpointer
    )
    
    # 대화 진행
    config = {"configurable": {"thread_id": "user_123"}}
    
    result = agent_persistent.invoke(
        {"messages": [{"role": "user", "content": "내 이메일은 hong@example.com이야."}]},
        config
    )
    print("저장 완료:", result["messages"][-1].content)

# 데이터베이스에서 다시 로드
with SqliteSaver.from_conn_string(DB_PATH) as checkpointer:
    agent_reload = create_agent(
        model,
        tools=[],
        checkpointer=checkpointer
    )
    
    # 같은 thread_id로 이전 대화 이어가기
    config = {"configurable": {"thread_id": "user_123"}}
    
    result = agent_reload.invoke(
        {"messages": [{"role": "user", "content": "내 이메일 주소가 뭐였지?"}]},
        config
    )
    print("\n재로드 후 응답:", result["messages"][-1].content)

저장 완료: 알겠습니다. 이메일 주소는 **hong@example.com**으로 기억할게요.

재로드 후 응답: 이메일 주소는 **hong@example.com**이었습니다.


## 3. 대화 이력 조회

Checkpointer를 통해 저장된 대화 이력을 조회할 수 있습니다.

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent = create_agent(
    model,
    tools=[],
    checkpointer=checkpointer
)

# 여러 대화 진행
config = {"configurable": {"thread_id": "test_thread"}}

agent.invoke(
    {"messages": [{"role": "user", "content": "안녕! 오늘 날씨가 좋네요."}]},
    config
)

agent.invoke(
    {"messages": [{"role": "user", "content": "점심으로 뭐 먹을까요?"}]},
    config
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "오늘 우리가 무슨 얘기했지?"}]},
    config
)

# 최종 응답 확인
result["messages"][-1].pretty_print()

# 전체 메시지 이력 확인
print("\n=== 전체 대화 이력 ===")
for i, msg in enumerate(result["messages"], 1):
    print(f"\n{i}. [{msg.type}] {msg.content[:100]}...")

================================== Ai Message ==================================

오늘은 이런 얘기를 했어요:

1. **날씨 얘기**
   - 당신이 “오늘 날씨가 좋네요”라고 했고,
   - 제가 “기분 좋은 하루네요”처럼 답했어요.

2. **점심 메뉴 얘기**
   - 당신이 “점심으로 뭐 먹을까요?”라고 물었고,
   - 제가 가볍게/든든하게/국물 있는 메뉴/기분 내는 메뉴로 몇 가지 추천했어요.

지금까지 대화는 **날씨와 점심 메뉴 추천**이었어요.

=== 전체 대화 이력 ===

1. [human] 안녕! 오늘 날씨가 좋네요....

2. [ai] 안녕하세요! 정말 좋은 날씨네요 ☀️  
기분 좋은 하루 보내고 계신가요?...

3. [human] 점심으로 뭐 먹을까요?...

4. [ai] 좋네요. 오늘 같은 날엔 가볍게도, 든든하게도 다 괜찮아요.

추천 몇 가지 드리면:
- **가볍게**: 샐러드, 쌀국수, 냉면
- **든든하게**: 비빔밥, 제육덮밥, 돈까스
-...

5. [human] 오늘 우리가 무슨 얘기했지?...

6. [ai] 오늘은 이런 얘기를 했어요:

1. **날씨 얘기**
   - 당신이 “오늘 날씨가 좋네요”라고 했고,
   - 제가 “기분 좋은 하루네요”처럼 답했어요.

2. **점심 메뉴 얘...


## 실습문제: thread 격리와 영구 메모리

본문에서 배운 checkpointer와 thread_id 개념을 확인해 보세요.

1. **thread 격리 확인**: `InMemorySaver` 에이전트를 만들고,
   thread `"user_a"`에서 `"내가 좋아하는 색은 파란색이야"`라고 말한 뒤 —
   같은 thread에서는 기억하는지, **다른 thread** `"user_b"`에서는 기억하지 못하는지 확인하세요. (1.2장 참고)
2. **영구 저장**: `SqliteSaver`(`my_memory.db`)로 에이전트를 만들어 좋아하는 음식을 알려주세요. (2.1장 참고)
3. **재로드**: checkpointer를 **새로 열어** 같은 thread_id로 질문해서,
   DB 파일에서 대화가 복원되는지 확인하세요.

---------------------------------
### 실전 예제: Flask 웹 애플리케이션

위에서 학습한 에이전트와 도구 사용법을 **Flask 웹 애플리케이션**으로 구현한 실전 예제를 참고하세요:

- **`Flask_WebUI/101_Tools_Agent.py`** - ReAct Agent 기반 Chatbot 구현
  - `create_agent`를 사용한 에이전트 생성
  - 도구를 활용한 실시간 검색 및 응답
  - 웹 UI를 통한 사용자 인터랙션
  - memory 를 이용한 대화 내용 저장